# Per-stage comparison across a batch

This notebook splits a **batch** into time-range stages and compares averaged
ion/compound intensities between them. Where
[05_peaks_by_stage](05_peaks_by_stage.ipynb) divides the intra-sample timeseries
of a *single sample* into stages, here the stages divide the *batch* timeline:
each stage is a contiguous time range that groups whole samples (e.g.
background → exposure → recovery, or day/night periods).

The workflow is:

1. Load the averaged peaks of every sample in the batch with `load_peaks()`.
2. Plot the batch timeseries to **visually identify** stage boundaries.
3. Define stages as `(start, end, name)` datetime tuples and assign samples to them.
4. Average ion/compound intensities across the samples within each stage and compare.
5. Normalise by each sample's TIC to compare relative changes between stages.


In [ ]:
from mascope_sdk import MascopeClient


mascope = MascopeClient(workspace="My Workspace")

### Load peaks for the batch

`load_peaks` returns the time-averaged peak list of every sample, enriched with
the sample's measurement start time (`datetime_utc`). That timestamp is the
batch time axis we will split into stages.


In [ ]:
# Adjust these to match your data
dataset = "My Dataset"
batch = "My Batch"

peaks = mascope.load_peaks(dataset=dataset, batches=batch)
peaks

In [ ]:
# Keep only matched peaks
matched = peaks[peaks["target_compound_formula"].notna()].copy()

# Composite label: shows the compound + ionization path that produced the ion formula.
# Two different compounds can yield the same ion formula via different mechanisms;
# this column makes each path visible in plots.
matched["ion_formula_path"] = (
    matched["target_compound_formula"] + matched["ionization_mechanism"]
)

print(
    f"Batch spans {peaks['datetime_utc'].min()} - {peaks['datetime_utc'].max()} "
    f"({peaks['sample_item_id'].nunique()} samples)"
)
matched

### Visualise the batch timeseries

Plot ion-level intensities per sample over the batch. Use this plot (and the
time span printed above) to decide where the stage boundaries should be.


In [ ]:
import plotly.express as px
import plotly.io as pio


pio.templates.default = "plotly_dark"  # or "plotly_white"

# Match score threshold (0 – 1)
ion_score_min = 0.9

ion_filtered = matched[matched["match_score_ion"] >= ion_score_min]

# Sum isotopes belonging to the same ion within each sample
ion_ts = (
    ion_filtered.groupby(
        ["datetime_utc", "sample_item_name", "ion_formula_path", "target_ion_formula"]
    )["height"]
    .sum()
    .reset_index()
    .sort_values("datetime_utc")
)

fig = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name", "target_ion_formula"],
    title=f"Ion-level intensities across batch {batch}",
)
fig.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig.show()

### Define stages

Based on the plot above, define the stages. Each stage is a tuple of
`(start, end, name)` where the boundaries are datetimes in **UTC**. Ends are
exclusive, so back-to-back stages can share a boundary without assigning a
sample to both.


In [ ]:
# Adjust boundaries based on the batch timeseries above (UTC)
stages = [
    ("2025-01-01 00:00", "2025-01-01 08:00", "background"),
    ("2025-01-01 08:00", "2025-01-01 16:00", "exposure"),
    ("2025-01-01 16:00", "2025-01-02 00:00", "recovery"),
]

Assign each sample's peaks to a stage. Stages may overlap (a sample then
contributes to every stage it falls in); samples outside all stages are
reported and left out.


In [ ]:
import pandas as pd


frames = []
for idx, (start, end, name) in enumerate(stages):
    t0 = pd.Timestamp(start, tz="UTC")
    t1 = pd.Timestamp(end, tz="UTC")
    sel = matched[(matched["datetime_utc"] >= t0) & (matched["datetime_utc"] < t1)]
    sel = sel.copy()
    sel["stage"] = idx
    sel["stage_name"] = name
    sel["t_start"] = t0
    sel["t_end"] = t1
    frames.append(sel)

staged = pd.concat(frames, ignore_index=True)

print(staged.groupby("stage_name", sort=False)["sample_item_id"].nunique().rename("samples"))

unassigned = set(matched["sample_item_id"]) - set(staged["sample_item_id"])
if unassigned:
    names = matched.loc[
        matched["sample_item_id"].isin(unassigned), "sample_item_name"
    ].unique()
    print(f"{len(unassigned)} sample(s) fall outside all stages: {sorted(names)}")

Visualize the stages


In [ ]:
colors = px.colors.qualitative.Plotly

fig2 = px.scatter(
    ion_ts,
    x="datetime_utc",
    y="height",
    color="ion_formula_path",
    hover_data=["sample_item_name"],
    title=f"Stages of batch {batch}",
)
for i, (start, end, label) in enumerate(stages):
    color = colors[i % len(colors)]
    fig2.add_vrect(
        x0=pd.Timestamp(start, tz="UTC"),
        x1=pd.Timestamp(end, tz="UTC"),
        fillcolor=color,
        opacity=0.15,
        line_width=1,
        line_color=color,
        annotation_text=label,
        annotation_position="top left",
    )
fig2.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig2.show()

### Stage averages

Averaging happens in two steps:

1. Within each sample, sum the intensities belonging to the same ion (or compound).
2. Within each stage, average those per-sample intensities across samples.

An ion that was not detected in a sample counts as **0** in that sample (the
per-sample grid is completed before averaging). Without this, stage averages
would be biased upward for ions that appear in only a few samples. Every sample
weighs equally in the stage average, regardless of its duration.

The helper below implements both steps; it is reused for ion-level,
compound-level, and normalised averages.


In [ ]:
def stage_average(df, level, value="height"):
    """Mean/std of per-sample summed intensities across the samples of each stage.

    ``level`` is the column defining the aggregation level (e.g.
    ``"ion_formula_path"`` or ``"target_compound_formula"``); ``value`` is the
    intensity column to aggregate. The (sample, level) grid is completed with
    zeros so undetected ions/compounds count as 0 in each sample.
    """
    per_sample = (
        df.groupby(["stage", "stage_name", "sample_item_name", level])[value]
        .sum()
        .reset_index()
    )
    wide = per_sample.pivot_table(
        index=["stage", "stage_name", "sample_item_name"],
        columns=level,
        values=value,
        fill_value=0,
    )
    return (
        wide.stack()
        .rename(value)
        .reset_index()
        .groupby(["stage", "stage_name", level])[value]
        .agg(mean="mean", std="std", n_samples="count")
        .reset_index()
        .sort_values("stage")
    )

#### Ion level


In [ ]:
# Match score threshold (0 – 1)
ion_score_min = 0.9

ion_stage = stage_average(
    staged[staged["match_score_ion"] >= ion_score_min], level="ion_formula_path"
)
ion_stage

In [ ]:
# Keep the strongest ions so the bar chart stays readable
top_n = 15
top_ions = (
    ion_stage.groupby("ion_formula_path")["mean"].max().nlargest(top_n).index
)

fig3 = px.bar(
    ion_stage[ion_stage["ion_formula_path"].isin(top_ions)],
    x="ion_formula_path",
    y="mean",
    color="stage_name",
    barmode="group",
    error_y="std",
    hover_data=["n_samples"],
    title=f"Mean ion intensity by stage (top {top_n} ions, error bars = std)",
)
fig3.update_layout(
    xaxis_title="Ion (compound + mechanism)", yaxis_title="Mean intensity [cps]"
)
fig3.show()

#### Compound level

Same idea one level up: sum all peaks belonging to the same compound within
each sample, then average across the samples of each stage.

**Note:** Different compounds can produce the same ion formula (via different
ionization mechanisms). A peak matching such an ion contributes to _both_
compounds, so compound-level intensities are not strictly additive across
compounds.


In [ ]:
# Match score threshold (0 – 1)
compound_score_min = 0.9

compound_stage = stage_average(
    staged[staged["match_score_compound"] >= compound_score_min],
    level="target_compound_formula",
)

top_compounds = (
    compound_stage.groupby("target_compound_formula")["mean"].max().nlargest(top_n).index
)

fig4 = px.bar(
    compound_stage[compound_stage["target_compound_formula"].isin(top_compounds)],
    x="target_compound_formula",
    y="mean",
    color="stage_name",
    barmode="group",
    error_y="std",
    hover_data=["n_samples"],
    title=f"Mean compound intensity by stage (top {top_n} compounds, error bars = std)",
)
fig4.update_layout(xaxis_title="Compound", yaxis_title="Mean intensity [cps]")
fig4.show()

### TIC-normalised stage averages

Raw stage means mix real concentration changes with drift in overall signal
strength (source performance, inlet transmission, ...). Dividing each sample's
intensities by that sample's **Total Ion Count** (sum of all peak heights, as
in [06_normalization](06_normalization.ipynb)) removes this variation before
the stage average is taken.

Normalising per sample *before* averaging keeps a sample with an unusually
high TIC from dominating the stage mean, which is why this is preferred over
dividing the stage mean by the stage-mean TIC.

> **Reagent-ion normalisation:** if your data has a paired RI batch, normalise
> by the reagent-ion signal instead: build `height_norm` following Method 2 of
> [06_normalization](06_normalization.ipynb), then reuse `stage_average` on it
> exactly as below.


In [ ]:
# TIC per sample (sum of unique peak heights, before match expansion)
tic = (
    peaks.drop_duplicates(subset=["sample_item_id", "peak_id"])
    .groupby("sample_item_id")["height"]
    .sum()
)

staged["height_norm"] = staged["height"] / staged["sample_item_id"].map(tic)

ion_stage_norm = stage_average(
    staged[staged["match_score_ion"] >= ion_score_min],
    level="ion_formula_path",
    value="height_norm",
)

fig5 = px.bar(
    ion_stage_norm[ion_stage_norm["ion_formula_path"].isin(top_ions)],
    x="ion_formula_path",
    y="mean",
    color="stage_name",
    barmode="group",
    error_y="std",
    hover_data=["n_samples"],
    title=f"TIC-normalised mean ion intensity by stage (top {top_n} ions)",
)
fig5.update_layout(
    xaxis_title="Ion (compound + mechanism)", yaxis_title="Mean height / TIC"
)
fig5.show()

### Fold change against a reference stage

Divide each stage's normalised mean by that of a reference stage (e.g. the
background) to see which ions respond. The log2 scale makes increases and
decreases symmetric: +1 means doubled, -1 means halved.


In [ ]:
import numpy as np


reference_stage = "background"  # adjust to one of your stage names

means = ion_stage_norm.pivot_table(
    index="ion_formula_path", columns="stage_name", values="mean"
)
# Zeros would make the ratio blow up / vanish; treat them as missing
fold_change = means.div(means[reference_stage].replace(0, np.nan), axis=0)
log2_fc = np.log2(fold_change.replace(0, np.nan))

# Order stages as defined and drop the reference column (identically 0)
stage_order = [name for _, _, name in stages if name != reference_stage]
log2_fc = log2_fc[stage_order].loc[top_ions.intersection(log2_fc.index)]

fig6 = px.imshow(
    log2_fc,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    aspect="auto",
    labels={"x": "Stage", "y": "Ion", "color": f"log2 vs {reference_stage}"},
    title=f"Ion fold change vs '{reference_stage}' stage (TIC-normalised)",
)
fig6.show()